In [1]:
# =============================================================================
# GUS02B: Cross Tables, Distributions & Data Quality-of-Life Functions
# =============================================================================
# This notebook demonstrates the v4.1+ cross table and QoL capabilities:
#   1. Load database with data (from GUS02A)
#   2. Build cross tables from loaded subject data
#   3. Inspect cross tables (DataFrame view, per year)
#   4. Aggregate cross tables across TERYTs
#   5. Aggregate raw data + joint/marginal distributions
#   6. Manually insert a cross table and deconstruct to raw data
#   7. QoL functions: subject_availability, get_subject_dataframe, etc.
#   8. Save/reload with cross table persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc
import random

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"Module version attributes: CrossTable={hasattr(gtdb, 'CrossTable')}, YEAR_RANGE_FULL={hasattr(gtdb, 'YEAR_RANGE_FULL')}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS
Module version attributes: CrossTable=True, YEAR_RANGE_FULL=True


In [2]:
# =============================================================================
# STEP 2: Load Database (with all data from GUS02A)
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_final.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()
print(f"\nData summary: {db.get_data_summary()}")

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  Database version: 4.2
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4561 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
  ✓ Records with data: 4533
  ✓ Records with population data: 4533
  ✓ Records with pop_class: 3411
GeoTERYT Database Summary (v3.0)
Total records:           4,561
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes

In [3]:
# Subject name mapping (needed for cross table building)
subject_names_dict = {
    'P2137': 'pop__age_sex',
    'P2884': 'pop__age', 'P2885': 'pop__educ', 'P2883': 'pop__sex', 'P2887': 'hh_size',
    'P2114': 'pop__age_sex', 'P2403': 'pop__age_educ', 'P2402': 'pop__sex_educ', 'P2871': 'hh_size',
    'P3304': 'pop__age_sex', 'P3311': 'pop__age_educ', 'P3309': 'pop__sex_educ', 'P3420': 'hh_size',
    'P4253': 'pop__age_sex', 'P4320': 'pop__age_educ', 'P4315': 'pop__sex_educ', 'P4287': 'hh_size'
}

# Add merged subjects (these were created in GUS02A)
merged_subjects = {sid: sid.replace('M_', '') for sid in db.get_data_summary()['subjects'] if sid.startswith('M_')}
subject_names_dict.update(merged_subjects)
print(f"Total subjects: {len(subject_names_dict)}")
print(f"  Raw: {sum(1 for s in subject_names_dict if not s.startswith('M_'))}")
print(f"  Merged: {sum(1 for s in subject_names_dict if s.startswith('M_'))}")

# Randomly select a record for inspection throughout this notebook
random.seed(42)
candidates = [r for r in db._records.values() if r.level == 6 and r.has_data and r.n_data_series > 50]
sample_rec = random.choice(candidates)
print(f"\nSample record for inspection: {sample_rec.teryt_id} - {sample_rec.name} ({sample_rec.kind})")
print(f"  Data series: {sample_rec.n_data_series}, Subjects: {sample_rec.list_subjects()}")

Total subjects: 21
  Raw: 17
  Merged: 4

Sample record for inspection: 1002102 - Strzelce (rural)
  Data series: 434, Subjects: ['P2114', 'P2884', 'P3304', 'P4287', 'P4315', 'M_pop__age_sex', 'P2883', 'M_hh_size', 'P2885', 'P2137', 'P2887', 'P4253', 'P2402', 'M_pop__sex_educ', 'P2871']


In [4]:
# =============================================================================
# STEP 4: Build Cross Tables from Loaded Data
# =============================================================================
# Build cross tables for ALL subjects (raw + merged).
# Cross tables are M-dimensional arrays (e.g., age x sex for P2137).

importlib.reload(gtdb)

for s, name in subject_names_dict.items():
    n_built = db.build_cross_tables(s, subject_name=name)
    
print(f"\n{'='*60}")
print("Cross table summary:")
ct_summary = db.get_cross_table_summary()
display(ct_summary)

  ✓ Built 4533 cross tables for subject P2137 (pop__age_sex)
  ✓ Built 3624 cross tables for subject P2884 (pop__age)
  ✓ Built 3624 cross tables for subject P2885 (pop__educ)
  ✓ Built 3624 cross tables for subject P2883 (pop__sex)
  ✓ Built 3624 cross tables for subject P2887 (hh_size)
  ✓ Built 3647 cross tables for subject P2114 (pop__age_sex)
  ✓ Built 380 cross tables for subject P2403 (pop__age_educ)
  ✓ Built 3647 cross tables for subject P2402 (pop__sex_educ)
  ✓ Built 3647 cross tables for subject P2871 (hh_size)
  ✓ Built 3700 cross tables for subject P3304 (pop__age_sex)
  ✓ Built 379 cross tables for subject P3311 (pop__age_educ)
  ✓ Built 379 cross tables for subject P3309 (pop__sex_educ)
  ✓ Built 379 cross tables for subject P3420 (hh_size)
  ✓ Built 3798 cross tables for subject P4253 (pop__age_sex)
  ✓ Built 380 cross tables for subject P4320 (pop__age_educ)
  ✓ Built 4195 cross tables for subject P4315 (pop__sex_educ)
  ✓ Built 3798 cross tables for subject P4287 (hh

,subject_name,n_records,dimensions,shape
subject_id,,,,
P2137,pop__age_sex,4533,n1 × n2,"(21, 3)"
P4315,pop__sex_educ,4195,n1 × n2,"(3, 10)"
M_pop__age_sex,pop__age_sex,4533,n1 × n2,"(19, 3)"
M_pop__sex_educ,pop__sex_educ,4275,n1 × n2,"(3, 10)"
P2403,pop__age_educ,380,n1 × n2,"(12, 8)"
P3311,pop__age_educ,379,n1 × n2,"(13, 9)"
P3309,pop__sex_educ,379,n1 × n2,"(3, 9)"
P3420,hh_size,379,n1,"(6,)"
P4320,pop__age_educ,380,n1 × n2,"(12, 9)"


In [5]:
# =============================================================================
# STEP 5: Inspect Cross Tables on Individual Records
# =============================================================================
# Inspect cross tables for the randomly selected record

rec = sample_rec
print(f"Record: {rec}")
print(f"Cross tables: {rec.list_cross_tables()}")

# Show cross table for a subject with 2 dimensions (e.g., P2137 if available)
inspect_sid = None
for sid in ['P2137', 'M_pop__age_sex', 'P2402']:
    ct_check = rec.get_cross_table(sid)
    if ct_check and ct_check.ndim >= 2:
        inspect_sid = sid
        break

if inspect_sid:
    ct = rec.get_cross_table(inspect_sid)
    print(f"\n{ct}")
    print(f"Dimensions: {ct.dim_names}")
    print(f"Labels: {ct.dim_labels}")
    print(f"Years with data: {ct.years_with_data}")
    
    # Show as DataFrame for the most recent year with data
    yr = ct.years_with_data[-1] if ct.years_with_data else 2020
    print(f"\n--- Cross table for {rec.name}, {inspect_sid}, year {yr} ---")
    display(ct.get_as_dataframe(yr))
else:
    print("No 2D cross table available for this record")

Record: TERYTRecord(1002102, Strzelce, years=1999-2024, no changes)
Cross tables: ['P2137', 'P2884', 'P2885', 'P2883', 'P2887', 'P2114', 'P2402', 'P2871', 'P3304', 'P4253', 'P4315', 'P4287', 'M_hh_size', 'M_pop__age_sex', 'M_pop__sex_educ']

CrossTable(P2137, dims=['n1', 'n2'], shape=(21, 3), 30 years with data [1995-2024])
Dimensions: ['n1', 'n2']
Labels: {'n1': ['0-14', '0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70 i więcej', '70-74', '75-79', '80-84', '85 i więcej', 'ogółem'], 'n2': ['kobiety', 'mężczyźni', 'ogółem']}
Years with data: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

--- Cross table for Strzelce, P2137, year 2024 ---


,kobiety,mężczyźni,ogółem
0-14,253.0,269.0,522.0
0-4,80.0,75.0,155.0
10-14,83.0,96.0,179.0
15-19,96.0,98.0,194.0
20-24,87.0,110.0,197.0
25-29,94.0,105.0,199.0
30-34,120.0,125.0,245.0
35-39,127.0,147.0,274.0
40-44,119.0,147.0,266.0
45-49,156.0,135.0,291.0


In [6]:
# =============================================================================
# STEP 6: Inspect Merged Cross Tables (unified census + BDL data)
# =============================================================================
# Check if the merged subjects have properly built cross tables

rec = sample_rec
merged_sids = [s for s in rec.list_cross_tables() if s.startswith('M_')]
print(f"Merged cross tables on {rec.name}: {merged_sids}")

for msid in merged_sids[:2]:
    ct_m = rec.get_cross_table(msid)
    if ct_m:
        print(f"\n{ct_m}")
        print(f"  Dims: {ct_m.dim_names}, Labels: {ct_m.dim_labels}")
        print(f"  Years with data: {ct_m.years_with_data}")
        if ct_m.years_with_data:
            yr = ct_m.years_with_data[-1]
            print(f"\n--- {msid} for {rec.name}, year {yr} ---")
            display(ct_m.get_as_dataframe(yr))

# Also show a raw census cross table if available
census_sids = [s for s in rec.list_cross_tables() if s.startswith('P2') and s not in ['P2137']]
for csid in census_sids[:1]:
    ct_c = rec.get_cross_table(csid)
    if ct_c and ct_c.years_with_data:
        yr = ct_c.years_with_data[0]
        print(f"\n--- Raw census: {csid} ({subject_names_dict.get(csid,'')}), year {yr} ---")
        display(ct_c.get_as_dataframe(yr))

Merged cross tables on Strzelce: ['M_hh_size', 'M_pop__age_sex', 'M_pop__sex_educ']

CrossTable(M_hh_size, dims=['n1'], shape=(4,), 1 years with data [1988-1988])
  Dims: ['n1'], Labels: {'n1': ['1', '2', '3-4', '5 i więcej']}
  Years with data: [1988]

--- M_hh_size for Strzelce, year 1988 ---


,value
1,202.0
2,259.0
3-4,488.0
5 i więcej,405.0



CrossTable(M_pop__age_sex, dims=['n1', 'n2'], shape=(19, 3), 30 years with data [1995-2024])
  Dims: ['n1', 'n2'], Labels: {'n1': ['0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85 i więcej', 'ogółem'], 'n2': ['kobiety', 'mężczyźni', 'ogółem']}
  Years with data: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

--- M_pop__age_sex for Strzelce, year 2024 ---


,kobiety,mężczyźni,ogółem
0-4,80.0,75.0,155.0
10-14,83.0,96.0,179.0
15-19,96.0,98.0,194.0
20-24,87.0,110.0,197.0
25-29,94.0,105.0,199.0
30-34,120.0,125.0,245.0
35-39,127.0,147.0,274.0
40-44,119.0,147.0,266.0
45-49,156.0,135.0,291.0
5-9,90.0,98.0,188.0



--- Raw census: P2884 (pop__age), year 1988 ---


,value
0-9,739.0
10-19,723.0
20-29,698.0
30-39,705.0
40-49,506.0
50-59,548.0
60 lat i więcej,944.0
ogółem,4865.0


In [7]:
# =============================================================================
# STEP 7: Aggregate Cross Tables Across TERYTs
# =============================================================================
# Aggregate P2137 (pop by age x sex) across all gminas in Malopolskie

malopolskie_gminas = db.get_gminas_in_voivodeship('12', year=2020)
teryt_ids = [r.teryt_id for r in malopolskie_gminas]
print(f"Aggregating {len(teryt_ids)} gminas in Malopolskie voivodeship...")

agg_ct = db.aggregate_cross_tables(teryt_ids, 'P2137')
if agg_ct:
    print(f"\nAggregated: {agg_ct}")
    print(f"\n--- Aggregated cross table for Malopolskie, P2137, year 2020 ---")
    display(agg_ct.get_as_dataframe(2020))
else:
    print("No cross tables found for aggregation")

Aggregating 182 gminas in Malopolskie voivodeship...

Aggregated: CrossTable(P2137, dims=['n1', 'n2'], shape=(21, 3), 30 years with data [1995-2024])

--- Aggregated cross table for Malopolskie, P2137, year 2020 ---


,kobiety,mężczyźni,ogółem
0-14,273493.0,289031.0,562524.0
0-4,91144.0,96675.0,187819.0
10-14,92054.0,96607.0,188661.0
15-19,81052.0,84661.0,165713.0
20-24,94734.0,96775.0,191509.0
25-29,118711.0,120241.0,238952.0
30-34,132348.0,135193.0,267541.0
35-39,144206.0,145298.0,289504.0
40-44,133945.0,135547.0,269492.0
45-49,118114.0,117241.0,235355.0


In [8]:
# =============================================================================
# STEP 7B: Joint and Marginal Distributions
# =============================================================================
# Use aggregate_data() and get_distribution() functions

# Aggregate raw data for Malopolskie, P2137, year 2020
print("=== Aggregate Data (P2137, Malopolskie 2020) ===")
agg_df = db.aggregate_data(malopolskie_gminas, 'P2137', 2020, agg_func='sum')
print(f"Aggregated data shape: {agg_df.shape}")
display(agg_df.head(10))

# Joint distribution: age group (n1) x gender (n2) for Malopolskie, year 2020
print("\n=== Joint Distribution (age group x gender) - Malopolskie 2020 ===")
joint = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                            row_category='n1', col_category='n2')
display(joint)

# Marginal distribution: by gender only
print("\n=== Marginal Distribution (by gender) - Malopolskie 2020 ===")
marginal_gender = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                                       col_category='n2')
display(marginal_gender)

# Marginal distribution: by age group only
print("\n=== Marginal Distribution (by age group) - Malopolskie 2020 ===")
marginal_age = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                                    row_category='n1')
display(marginal_age)

=== Aggregate Data (P2137, Malopolskie 2020) ===
Aggregated data shape: (63, 4)


,variable_id,n1,n2,value
0,454046,0-14,kobiety,273493.0
1,454047,0-14,mężczyźni,289031.0
2,454048,0-14,ogółem,562524.0
3,47693,60-64,kobiety,113356.0
4,47694,25-29,ogółem,238952.0
5,47695,30-34,kobiety,132348.0
6,47696,25-29,kobiety,118711.0
7,47698,40-44,kobiety,133945.0
8,47701,35-39,ogółem,289504.0
9,47702,55-59,kobiety,103188.0



=== Joint Distribution (age group x gender) - Malopolskie 2020 ===


n2,kobiety,mężczyźni,ogółem
n1,,,
0-14,273493.0,289031.0,562524.0
0-4,91144.0,96675.0,187819.0
10-14,92054.0,96607.0,188661.0
15-19,81052.0,84661.0,165713.0
20-24,94734.0,96775.0,191509.0
25-29,118711.0,120241.0,238952.0
30-34,132348.0,135193.0,267541.0
35-39,144206.0,145298.0,289504.0
40-44,133945.0,135547.0,269492.0



=== Marginal Distribution (by gender) - Malopolskie 2020 ===


n2
kobiety      4051626.0
mężczyźni    3768139.0
ogółem       7819765.0
Name: value, dtype: float64


=== Marginal Distribution (by age group) - Malopolskie 2020 ===


n1
0-14           1125048.0
0-4             375638.0
10-14           377322.0
15-19           331426.0
20-24           383018.0
25-29           477904.0
30-34           535082.0
35-39           579008.0
40-44           538984.0
45-49           470710.0
5-9             372088.0
50-54           409220.0
55-59           404648.0
60-64           432244.0
65-69           394378.0
70 i więcej     783714.0
70-74           301318.0
75-79           181720.0
80-84           157408.0
85 i więcej     143268.0
ogółem         6865384.0
Name: value, dtype: float64

In [9]:
# =============================================================================
# STEP 8: Manual Cross Table Insertion and Deconstruction
# =============================================================================
# Demonstrate: manually insert a cross table, then deconstruct it back to data

rec = sample_rec

# Insert a 2D cross table manually (e.g., education x sex)
manual_table = np.array([
    [100, 200, 300],  # primary: total, male, female
    [150, 250, 350],  # secondary
    [200, 300, 400],  # tertiary
], dtype=float)

rec.insert_cross_table_year(
    subject_id='XTEST',
    year=2020,
    table=manual_table,
    subject_name='test_educ_sex',
    dim_names=['n1', 'n2'],
    dim_labels={'n1': ['primary', 'secondary', 'tertiary'],
                'n2': ['total', 'male', 'female']}
)

ct_manual = rec.get_cross_table('XTEST')
print(f"Manually inserted: {ct_manual}")
display(ct_manual.get_as_dataframe(2020))

# Deconstruct back to raw data points
n_pts = rec.deconstruct_cross_table('XTEST', year=2020, source_type='Manual')
print(f"\nDeconstructed to {n_pts} data points")

# Show the data points that were created
manual_series = rec.get_data_by_subject('XTEST')
for key, series in list(manual_series.items())[:5]:
    print(f"  {key}: categories={series.categories}, val@2020={series.get_value(2020)}")
print(f"  ... ({len(manual_series)} series total)")

Manually inserted: CrossTable(XTEST, dims=['n1', 'n2'], shape=(3, 3), 1 years with data [2020-2020])


,total,male,female
primary,100.0,200.0,300.0
secondary,150.0,250.0,350.0
tertiary,200.0,300.0,400.0



Deconstructed to 9 data points
  ('Manual', 'XTEST', 'primary|total'): categories={'n1': 'primary', 'n2': 'total'}, val@2020=100.0
  ('Manual', 'XTEST', 'primary|male'): categories={'n1': 'primary', 'n2': 'male'}, val@2020=200.0
  ('Manual', 'XTEST', 'primary|female'): categories={'n1': 'primary', 'n2': 'female'}, val@2020=300.0
  ('Manual', 'XTEST', 'secondary|total'): categories={'n1': 'secondary', 'n2': 'total'}, val@2020=150.0
  ('Manual', 'XTEST', 'secondary|male'): categories={'n1': 'secondary', 'n2': 'male'}, val@2020=250.0
  ... (9 series total)


In [10]:
# =============================================================================
# STEP 10: QoL - Subject Availability
# =============================================================================
# Check which subjects are available across all gminas

avail = db.subject_availability(level=6, mode='years')
print(f"Availability matrix shape: {avail.shape}")
print(f"Columns: {list(avail.columns)}\n")

# Show summary: how many gminas have data for each subject
print("Records with data per subject:")
for col in avail.columns:
    if col == 'name':
        continue
    n_true = avail[col].sum() if avail[col].dtype == bool else (avail[col] != '').sum()
    print(f"  {col} ({subject_names_dict.get(col, '')}): {n_true}")

# Show first few rows
display(avail.head(10))

Availability matrix shape: (4162, 17)
Columns: ['name', 'M_hh_size', 'M_pop__age_sex', 'M_pop__sex_educ', 'P2114', 'P2137', 'P2402', 'P2871', 'P2883', 'P2884', 'P2885', 'P2887', 'P3304', 'P4253', 'P4287', 'P4315', 'XTEST']

Records with data per subject:
  M_hh_size (hh_size): 3624
  M_pop__age_sex (pop__age_sex): 4134
  M_pop__sex_educ (pop__sex_educ): 3878
  P2114 (pop__age_sex): 3647
  P2137 (pop__age_sex): 4134
  P2402 (pop__sex_educ): 3647
  P2871 (hh_size): 3647
  P2883 (pop__sex): 3624
  P2884 (pop__age): 3624
  P2885 (pop__educ): 3624
  P2887 (hh_size): 3624
  P3304 (pop__age_sex): 3700
  P4253 (pop__age_sex): 3798
  P4287 (hh_size): 3798
  P4315 (pop__sex_educ): 3798
  XTEST (): 1


,name,M_hh_size,M_pop__age_sex,M_pop__sex_educ,P2114,P2137,P2402,P2871,P2883,P2884,P2885,P2887,P3304,P4253,P4287,P4315,XTEST
teryt_id,,,,,,,,,,,,,,,,,
0201011,Bolesławiec,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201022,Bolesławiec,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201032,Gromadka,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201043,Nowogrodziec,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201044,Nowogrodziec,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201045,Nowogrodziec,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201052,Osiecznica,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0201062,Warta Bolesławiecka,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,
0202011,Bielawa,1988,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...","2002,2021",2002,"1995,1996,1997,1998,1999,2000,2001,2002,2003,2...",2002,2002,1988,1988,1988,1988,2011,2021,2021,2021,


In [11]:
# =============================================================================
# STEP 11: QoL - Get Subject DataFrame
# =============================================================================
# Reconstruct a full flat DataFrame from loaded subject data (reverse of load_subject_data)

# Example: get all P2137 data for a single TERYT
df_single = db.get_subject_dataframe('P2137', teryt_id=sample_rec.teryt_id)
print(f"P2137 data for {sample_rec.name} ({sample_rec.teryt_id}), all years:")
print(f"  Shape: {df_single.shape}")
display(df_single.head(20))

# Get for a specific year across all records
df_year = db.get_subject_dataframe('P2137', year=2020)
print(f"\nP2137 data for year 2020, all records:")
print(f"  Shape: {df_year.shape}")
display(df_year.head(10))

P2137 data for Strzelce (1002102), all years:
  Shape: (1752, 9)


,teryt_id,name,source_type,subject_id,variable_id,year,value,n1,n2
0,1002102,Strzelce,BDL,P2137,47693,1995,101.0,60-64,kobiety
1,1002102,Strzelce,BDL,P2137,47693,1996,107.0,60-64,kobiety
2,1002102,Strzelce,BDL,P2137,47693,1997,104.0,60-64,kobiety
3,1002102,Strzelce,BDL,P2137,47693,1998,108.0,60-64,kobiety
4,1002102,Strzelce,BDL,P2137,47693,1999,123.0,60-64,kobiety
5,1002102,Strzelce,BDL,P2137,47693,2000,125.0,60-64,kobiety
6,1002102,Strzelce,BDL,P2137,47693,2001,119.0,60-64,kobiety
7,1002102,Strzelce,BDL,P2137,47693,2002,115.0,60-64,kobiety
8,1002102,Strzelce,BDL,P2137,47693,2003,103.0,60-64,kobiety
9,1002102,Strzelce,BDL,P2137,47693,2004,91.0,60-64,kobiety



P2137 data for year 2020, all records:
  Shape: (263025, 9)


,teryt_id,name,source_type,subject_id,variable_id,year,value,n1,n2
0,0200000,DOLNOŚLĄSKIE,BDL,P2137,47693,2020,22489.0,60-64,kobiety
1,0200000,DOLNOŚLĄSKIE,BDL,P2137,47694,2020,59036.0,25-29,ogółem
2,0200000,DOLNOŚLĄSKIE,BDL,P2137,47695,2020,32391.0,30-34,kobiety
3,0200000,DOLNOŚLĄSKIE,BDL,P2137,47696,2020,30437.0,25-29,kobiety
4,0200000,DOLNOŚLĄSKIE,BDL,P2137,47698,2020,26974.0,40-44,kobiety
5,0200000,DOLNOŚLĄSKIE,BDL,P2137,47701,2020,66397.0,35-39,ogółem
6,0200000,DOLNOŚLĄSKIE,BDL,P2137,47702,2020,16663.0,55-59,kobiety
7,0200000,DOLNOŚLĄSKIE,BDL,P2137,47706,2020,15250.0,50-54,mężczyźni
8,0200000,DOLNOŚLĄSKIE,BDL,P2137,47707,2020,54017.0,40-44,ogółem
9,0200000,DOLNOŚLĄSKIE,BDL,P2137,47711,2020,19175.0,20-24,mężczyźni


In [12]:
# =============================================================================
# STEP 12: QoL - Get Variable Values (quick population table)
# =============================================================================
# Get all values of P2137 for year 2020 across gminas

pop_2020 = db.get_variable_values('P2137', year=2020, level=6)
print(f"Population data for year 2020, gmina level:")
print(f"  Shape: {pop_2020.shape}")
print(f"  Columns: {list(pop_2020.columns)[:10]}...")
display(pop_2020.head(10))

Population data for year 2020, gmina level:
  Shape: (3778, 64)
  Columns: ['name', '60-64/kobiety', '25-29/ogółem', '30-34/kobiety', '25-29/kobiety', '40-44/kobiety', '35-39/ogółem', '55-59/kobiety', '50-54/mężczyźni', '40-44/ogółem']...


,name,60-64/kobiety,25-29/ogółem,30-34/kobiety,25-29/kobiety,40-44/kobiety,35-39/ogółem,55-59/kobiety,50-54/mężczyźni,40-44/ogółem,...,75-79/mężczyźni,80-84/mężczyźni,85 i więcej/mężczyźni,70-74/ogółem,75-79/ogółem,80-84/ogółem,85 i więcej/ogółem,0-14/kobiety,0-14/mężczyźni,0-14/ogółem
teryt_id,,,,,,,,,,,,,,,,,,,,,
0201011,Bolesławiec,1717.0,2027.0,1342.0,1065.0,1554.0,3074.0,1165.0,1030.0,3058.0,...,425.0,335.0,267.0,2412.0,1220.0,1053.0,965.0,2425.0,2573.0,4998.0
0201022,Bolesławiec,510.0,934.0,542.0,456.0,604.0,1217.0,493.0,477.0,1252.0,...,111.0,71.0,37.0,561.0,262.0,228.0,170.0,1263.0,1365.0,2628.0
0201032,Gromadka,196.0,337.0,172.0,156.0,203.0,395.0,177.0,145.0,403.0,...,43.0,36.0,24.0,236.0,114.0,114.0,107.0,385.0,379.0,764.0
0201043,Nowogrodziec,538.0,947.0,564.0,459.0,596.0,1277.0,508.0,441.0,1213.0,...,114.0,96.0,45.0,522.0,309.0,252.0,236.0,1193.0,1315.0,2508.0
0201044,Nowogrodziec,173.0,247.0,170.0,121.0,186.0,358.0,179.0,99.0,359.0,...,33.0,27.0,12.0,177.0,88.0,82.0,63.0,313.0,347.0,660.0
0201045,Nowogrodziec,365.0,700.0,394.0,338.0,410.0,919.0,329.0,342.0,854.0,...,81.0,69.0,33.0,345.0,221.0,170.0,173.0,880.0,968.0,1848.0
0201052,Osiecznica,235.0,502.0,277.0,242.0,299.0,633.0,220.0,218.0,611.0,...,51.0,37.0,30.0,271.0,118.0,117.0,117.0,612.0,642.0,1254.0
0201062,Warta Bolesławiecka,339.0,471.0,306.0,238.0,340.0,762.0,274.0,258.0,692.0,...,61.0,41.0,35.0,330.0,163.0,132.0,130.0,720.0,688.0,1408.0
0202011,Bielawa,1292.0,1536.0,947.0,763.0,1190.0,2441.0,1023.0,856.0,2412.0,...,276.0,203.0,228.0,1724.0,743.0,775.0,892.0,1782.0,1857.0,3639.0


In [13]:
# =============================================================================
# STEP 12: Save & Reload with Cross Table Persistence
# =============================================================================
# Clean up the test cross table before saving
rec = sample_rec
if 'XTEST' in rec.cross_tables:
    del rec.cross_tables['XTEST']
test_keys = [k for k in rec.data.keys() if k[1] == 'XTEST']
for k in test_keys:
    del rec.data[k]

save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

# Reload and verify
db2 = gtdb.load_complete_database(save_path)
summary2 = db2.get_data_summary()
print(f"\nAfter reload: {summary2['records_with_data']} records with data, "
      f"{summary2['total_data_points']:,} total points")

# Check cross tables survived
ct_summary2 = db2.get_cross_table_summary()
print(f"\nCross tables after reload:")
display(ct_summary2)

# Spot check: sample record
rec2 = db2.get_by_teryt_id(sample_rec.teryt_id)
if rec2:
    ct_check = rec2.get_cross_table('P2137') or rec2.get_cross_table('P1336')
    if ct_check:
        yr = ct_check.years_with_data[-1] if ct_check.years_with_data else 2020
        print(f"\n{rec2.name} {ct_check.subject_id} after reload: {ct_check}")
        display(ct_check.get_as_dataframe(yr))

gc.collect()

Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  ✓ Saved 4561 records
  ✓ Records with data: 4533
  ✓ Records with cross tables: 4533
  ✓ File size: 2278.2 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  Database version: 4.2
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4561 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
  ✓ Records with data: 4533
  ✓ Records with cross tables: 4533
  ✓ Records w

,subject_name,n_records,dimensions,shape
subject_id,,,,
P2137,pop__age_sex,4533,n1 × n2,"(21, 3)"
P4315,pop__sex_educ,4195,n1 × n2,"(3, 10)"
M_pop__age_sex,pop__age_sex,4533,n1 × n2,"(19, 3)"
M_pop__sex_educ,pop__sex_educ,4275,n1 × n2,"(3, 10)"
P2403,pop__age_educ,380,n1 × n2,"(12, 8)"
P3311,pop__age_educ,379,n1 × n2,"(13, 9)"
P3309,pop__sex_educ,379,n1 × n2,"(3, 9)"
P3420,hh_size,379,n1,"(6,)"
P4320,pop__age_educ,380,n1 × n2,"(12, 9)"



Strzelce P2137 after reload: CrossTable(P2137, dims=['n1', 'n2'], shape=(21, 3), 30 years with data [1995-2024])


,kobiety,mężczyźni,ogółem
0-14,253.0,269.0,522.0
0-4,80.0,75.0,155.0
10-14,83.0,96.0,179.0
15-19,96.0,98.0,194.0
20-24,87.0,110.0,197.0
25-29,94.0,105.0,199.0
30-34,120.0,125.0,245.0
35-39,127.0,147.0,274.0
40-44,119.0,147.0,266.0
45-49,156.0,135.0,291.0


0